# RAG Pipeline — Document Assistant

**Author:** Ahmed Niazy  |  **Domain:** Pharmacy & pharmaceutical drug information

This notebook builds the full RAG pipeline:
**load → clean → chunk → embed → store → retrieve → generate → evaluate → export.**



## 0. Setup

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

# --- Paths (work whether Jupyter was started in the project root or in notebooks/) ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"                       # your source documents
EXPORT_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"  # the backend loads this folder

# --- Settings you can tune ---
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 250
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "documents"
OLLAMA_MODEL = "llama3.2"
TOP_K = 6
MIN_SCORE = 0.35

print("Project root:", PROJECT_ROOT)
print("Raw folder exists:", RAW_DIR.exists())

Project root: d:\rag-assistant-starter-kit\rag-assistant-project
Raw folder exists: True


## 1. Load & Inspect

In [61]:
files = sorted(p for p in RAW_DIR.iterdir() if p.suffix.lower() in {".pdf", ".txt", ".md"})
print(f"Found {len(files)} files")

pages = []    # one item per PDF page (or per text file)
failed = []   # files that could not be parsed

for path in files:
    try:
        if path.suffix.lower() == ".pdf":
            reader = PdfReader(str(path))
            for page_number, page in enumerate(reader.pages, start=1):
                pages.append({"source": path.name, "page": page_number, "text": page.extract_text() or ""})
        else:
            pages.append({"source": path.name, "page": 1, "text": path.read_text(encoding="utf-8", errors="ignore")})
    except Exception as e:
        failed.append((path.name, str(e)))

assert len(pages) > 0, "No documents loaded - put PDFs/TXT files into data/raw/ first!"
print(f"Loaded {len(pages)} pages/sections from {len(files) - len(failed)} files")
print("Files that failed to parse:", failed if failed else "none")

Found 10 files
Loaded 265 pages/sections from 10 files
Files that failed to parse: none


In [62]:
df_pages = pd.DataFrame(pages)
df_pages["chars"] = df_pages["text"].str.len()

summary = df_pages.groupby("source").agg(
    pages=("page", "count"),
    total_chars=("chars", "sum"),
    near_empty_pages=("chars", lambda s: int((s < 50).sum())),
)
display(summary)
print("Total pages:", len(df_pages))
print("Near-empty pages (possibly scanned images that need OCR):", int((df_pages["chars"] < 50).sum()))

,pages,total_chars,near_empty_pages
source,,,
Amlodipine_label.pdf,18,41789,0
Amoxicillin_label.pdf,26,57109,0
Atorvastatin_label.pdf,74,148529,0
Azithromycin_label.pdf,16,53404,0
Ciprofloxacin_label.pdf,20,35207,0
Lisinopril_label.pdf,18,58706,0
Losartan_label.pdf,39,75925,0
Metformin_label.pdf,46,104154,0
Omeprazole_label.pdf,6,5158,1


Total pages: 265
Near-empty pages (possibly scanned images that need OCR): 1


In [63]:
# Look at some raw text with your own eyes - what is messy?
sample = df_pages.iloc[min(2, len(df_pages) - 1)]
print(sample["source"], "- page", sample["page"])
print("-" * 60)
print(sample["text"][:800])

Amlodipine_label.pdf - page 3
------------------------------------------------------------
In hypertensive patients with normal renal function, therapeutic doses of amlodipine resulted in a
decrease in renal vascular resistance and an increase in glomerular filtration rate and effective renal
plasma flow without change in filtration fraction or proteinuria.
As with other calcium channel blockers, hemodynamic measurements of cardiac function at rest and
during exercise (or pacing) in patients with normal ventricular function treated with amlodipine have
generally demonstrated a small increase in cardiac index without significant influence on dP/dt or on left
ventricular end diastolic pressure or volume. In hemodynamic studies, amlodipine has not been
associated with a negative inotropic effect when administered in the therapeutic dose range to intact
animals and man, even when co


**Data inspection (fill this in with YOUR numbers):**

- Number of documents: **10**  |  Total pages: **265**
- Formats: PDF (TXT)
- Files that failed to parse / need OCR: **none** 
- Messy things I noticed: broken words at line ends, page numbers and headers repeated on every page, tables that came out as jumbled text._

### 1.1 Cleaning

In [64]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")                     # null characters
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)         # re-join words split at line ends: "infor-\nmation"
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)         # single line breaks -> spaces
    text = re.sub(r"[ \t]+", " ", text)                   # many spaces -> one
    text = re.sub(r"\n{2,}", "\n\n", text)                # many blank lines -> one
    return text.strip()


for p in pages:
    p["text"] = clean_text(p["text"])

pages = [p for p in pages if len(p["text"]) >= 50]      # drop empty / near-empty pages
print("Pages kept after cleaning:", len(pages))
print(pages[0]["text"][:500])

Pages kept after cleaning: 264
AMLODIPINE- amlodipine besylate tablet CARACO PHARMACEUTICAL LABORATORIES, LTD. ---------- DESCRIPTION Amlodipine besylate is the besylate salt of amlodipine, a long-acting calcium channel blocker. Amlodipine besylate is chemically described as 3-Ethyl-5-methyl (±)-2-[(2-aminoethoxy)methyl]-4-(2chlorophenyl)-1,4-dihydro-6-methyl-3,5-pyridinedicarboxylate, monobenzene-sulphonate. Its molecular formula is C H CIN O •C H O S, and its structural formula is: Amlodipine besylate is a white crystalline


## 2. Chunking Strategy

In [65]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Fixed-size chunks with overlap. We try to end each chunk at a sentence end (or at least a space)
    so we do not cut words in half."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            window = text[start:end]
            cut = max(window.rfind(". "), window.rfind("? "), window.rfind("! "), window.rfind("\n"))
            if cut > chunk_size * 0.5:            # found a sentence end in the second half of the window
                end = start + cut + 1
            else:
                space = window.rfind(" ")
                if space > chunk_size * 0.5:
                    end = start + space
        piece = text[start:end].strip()
        if len(piece) > 50:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)     # step back by `overlap` characters
    return chunks


chunks = []
for p in pages:
    for i, piece in enumerate(chunk_text(p["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
        chunks.append({
            "id": f'{p["source"]}::p{p["page"]}::c{i}',
            "text": piece,
            "source": p["source"],
            "page": p["page"],
            "chunk_index": i,
        })

lengths = pd.Series([len(c["text"]) for c in chunks])
print("Total chunks:", len(chunks))
display(lengths.describe().round(1))
print("\nExample chunk:\n", chunks[0]["text"])

Total chunks: 730


count     730.0
mean      950.0
std       275.4
min        85.0
25%       838.8
50%      1073.0
75%      1150.8
max      1200.0
dtype: float64


Example chunk:
 AMLODIPINE- amlodipine besylate tablet CARACO PHARMACEUTICAL LABORATORIES, LTD. ---------- DESCRIPTION Amlodipine besylate is the besylate salt of amlodipine, a long-acting calcium channel blocker. Amlodipine besylate is chemically described as 3-Ethyl-5-methyl (±)-2-[(2-aminoethoxy)methyl]-4-(2chlorophenyl)-1,4-dihydro-6-methyl-3,5-pyridinedicarboxylate, monobenzene-sulphonate. Its molecular formula is C H CIN O •C H O S, and its structural formula is: Amlodipine besylate is a white crystalline powder with a molecular weight of 567.1. It is slightly soluble in water and sparingly soluble in ethanol. Amlodipine besylate tablets are formulated as white tablets equivalent to 2.5 mg, 5 mg or 10 mg of amlodipine for oral administration. In addition to the active ingredient, amlodipine besylate, each tablet contains the following Inactive Ingredients: dibasic calcium phosphate (anhydrous), butylated hydroxytoluene, microcrystalline cellulose, sodium starch glycolate, magnes

**Why these chunk settings? (rewrite in your own words)**

I chose chunks of about **800 characters (~130 words)** with **150 characters of overlap** because:

- Smaller chunks (e.g. 200 chars) are very precise but often lose the context needed to answer.
- Much bigger chunks (e.g. 3000 chars) mix several topics, so the retrieved text is noisy and the small local LLM gets confused.
- The overlap makes sure a sentence that falls on a boundary still appears whole in at least one chunk.
- Chunks never cross page boundaries, so every chunk has an exact page number for citations.



## 3. Embeddings & Vector Store

In [66]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)   # first run downloads ~90 MB

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
print("Embedding matrix shape:", embeddings.shape)   # (number of chunks, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Embedding matrix shape: (730, 384)


In [67]:
import chromadb

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(EXPORT_DIR))   # saves to disk automatically

# Start from a clean collection so re-running the notebook never creates duplicates
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

BATCH = 500
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embeddings[i:i + BATCH].tolist(),
        metadatas=[{"source": c["source"], "page": c["page"], "chunk_index": c["chunk_index"]} for c in batch],
    )

print("Chunks in collection:", collection.count())

Chunks in collection: 730


In [68]:
# Prove it is really saved on disk: open a brand-new client and count again
check = chromadb.PersistentClient(path=str(EXPORT_DIR)).get_collection(COLLECTION_NAME)
print("Chunks stored on disk:", check.count())

Chunks stored on disk: 730


## 4. Retrieval & Prompting

In [69]:
def retrieve(question: str, top_k: int = TOP_K) -> list[dict]:
    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, meta, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        hits.append({"text": text, "source": meta["source"], "page": meta["page"], "score": 1 - dist})
    return hits

### 4.1 Test questions


In [70]:
TEST_SET = [
    {
        "question": "What are the Drug/Laboratory Test Interactions of Amoxicillin?",
        "expected_source": "Amoxicillin_label.pdf"
    },
    {
        "question": "What are the clinical considerations in lactations when using Azithromycin?",
        "expected_source": "Azithromycin_label.pdf"
    },
    {
        "question": "What are the contraindications of metformin?",
        "expected_source": "Metformin_label.pdf"
    },
    {
        "question": "What are the common adverse reactions of amlodipine?",
        "expected_source": "Amlodipine_label.pdf"
    },
    {
        "question": "What are the contraindications of atorvastatin?",
        "expected_source": "Atorvastatin_label.pdf"
    },
    {
        "question": "What are the warnings and precautions associated with ciprofloxacin?",
        "expected_source": "Ciprofloxacin_label.pdf"
    },
    {
        "question": "What are the contraindications of lisinopril?",
        "expected_source": "Lisinopril_label.pdf"
    },
    {
        "question": "What are the contraindications of losartan?",
        "expected_source": "Losartan_label.pdf"
    },
    {
        "question": "What are the indications and usage of omeprazole?",
        "expected_source": "Omeprazole_label.pdf"
    },
    {
        "question": "What are the common adverse reactions of sertraline?",
        "expected_source": "Sertraline_label.pdf"
    },

    # Out-of-scope questions
    {
        "question": "What is the recommended dose of warfarin?",
        "expected_source": None
    },
    {
        "question": "What is the recipe for chocolate cake?",
        "expected_source": None
    },
]

for item in TEST_SET:
    hits = retrieve(item["question"])

    print("Q:", item["question"])

    for h in hits[:3]:
        print(
            f'   {h["score"]:.2f} | '
            f'{h["source"]} p.{h["page"]} | '
            f'{h["text"][:90]!r}'
        )

    print()

Q: What are the Drug/Laboratory Test Interactions of Amoxicillin?
   0.63 | Amoxicillin_label.pdf p.2 | '64) Administered at the start of a light meal. Mean values of 24 normal volunteers. Peak c'
   0.61 | Amoxicillin_label.pdf p.1 | 'AMOXICILLIN- amoxicillin capsule AMOXICILLIN- amoxicillin tablet, film coated AMOXICILLIN-'
   0.58 | Amoxicillin_label.pdf p.2 | 'not been performed with the 200 mg and 500 mg formulations. Amoxicillin diffuses readily i'

Q: What are the clinical considerations in lactations when using Azithromycin?
   0.77 | Azithromycin_label.pdf p.8 | 'man milk [see Data ] . Non-serious adverse reactions have been reported in breastfed infan'
   0.74 | Azithromycin_label.pdf p.8 | 'well as 2 and 4 weeks postpartum revealed the presence of azithromycin in breastmilk up to'
   0.69 | Azithromycin_label.pdf p.8 | 'all groups; no evidence of fetotoxicity or teratogenicity was observed at these doses, the'

Q: What are the contraindications of metformin?
   0.60 | Metfor

In [71]:
NOT_FOUND_MESSAGE = "I could not find this in the provided documents."

SYSTEM_PROMPT = f"""You are a document assistant. Answer the user's question using ONLY the context provided.

Rules:
- If the answer is not in the context, reply exactly: "{NOT_FOUND_MESSAGE}"
- After each statement, cite the context block(s) it came from, like [1] or [2].
- Never use outside knowledge. Never invent facts.
- Keep the answer clear and concise."""


def build_prompt(question: str, hits: list[dict]) -> str:
    blocks = [f"[{i}] (source: {h['source']}, page {h['page']})\n{h['text']}" for i, h in enumerate(hits, start=1)]
    context = "\n\n".join(blocks)
    return f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer (with citations):"


def format_sources(hits: list[dict]) -> list[str]:
    return [f"[{i}] {h['source']} (page {h['page']})" for i, h in enumerate(hits, start=1)]

In [72]:
import ollama


def generate(question: str, hits: list[dict]) -> str:
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(question, hits)},
        ],
        options={"temperature": 0.1},
    )
    return response["message"]["content"].strip()


def ask(question: str):
    hits = retrieve(question)
    # Guardrail: if nothing relevant was retrieved, do not let the LLM improvise
    if not hits or hits[0]["score"] < MIN_SCORE:
        return NOT_FOUND_MESSAGE, hits
    return generate(question, hits), hits

In [73]:
# Test all questions end-to-end

for item in TEST_SET:
    answer, hits = ask(item["question"])

    print("=" * 100)
    print("QUESTION:")
    print(item["question"])

    print("\nANSWER:")
    print(answer)

    print("\nTOP SOURCE:")
    if hits:
        print(
            f'{hits[0]["source"]} | '
            f'page {hits[0]["page"]} | '
            f'score {hits[0]["score"]:.2f}'
        )
    else:
        print("No relevant source found.")

    print()

QUESTION:
What are the Drug/Laboratory Test Interactions of Amoxicillin?

ANSWER:
Unfortunately, I could not find this information in the provided documents.

TOP SOURCE:
Amoxicillin_label.pdf | page 2 | score 0.63

QUESTION:
What are the clinical considerations in lactations when using Azithromycin?

ANSWER:
According to the provided context, the clinical considerations in lactations when using Azithromycin are to advise women to monitor the breastfed infant for diarrhea, vomiting, or rash. [1]

TOP SOURCE:
Azithromycin_label.pdf | page 8 | score 0.77

QUESTION:
What are the contraindications of metformin?

ANSWER:
The contraindications of metformin are not explicitly stated in the provided documents. However, the documents do provide information on precautions and warnings, such as the risk of lactic acidosis [5] and the need for dose adjustments in patients with renal impairment [4]. 

I could not find this in the provided documents.

TOP SOURCE:
Metformin_label.pdf | page 36 | scor

## 5. Vision Component

_Core Track: not applicable — this project is text-only. (Not Extended Track, No YOLO model and no detections feed into the prompt )_

## 6. Evaluation

In [74]:
rows = []

for item in TEST_SET:
    answer, hits = ask(item["question"])

    retrieved_sources = sorted({
        h["source"] for h in hits
    })

    if item["expected_source"] is None:
        retrieval_ok = "n/a (out of scope)"
    else:
        retrieval_ok = (
            "yes"
            if item["expected_source"] in retrieved_sources
            else "no"
        )

    rows.append({
        "question": item["question"],
        "retrieved_source": ", ".join(retrieved_sources),
        "retrieval_relevant": retrieval_ok,
        "answer": answer,
    })

eval_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 200)

display(eval_df)

,question,retrieved_source,retrieval_relevant,answer
0,What are the Drug/Laboratory Test Interactions of Amoxicillin?,Amoxicillin_label.pdf,yes,"Unfortunately, I could not find any information on Drug/Laboratory Test Interactions of Amoxicillin in the provided documents."
1,What are the clinical considerations in lactations when using Azithromycin?,Azithromycin_label.pdf,yes,"According to the provided context, the clinical considerations in lactations when using Azithromycin are to advise women to monitor the breastfed infant for diarrhea, vomiting, or rash. [1]"
2,What are the contraindications of metformin?,Metformin_label.pdf,yes,"The contraindications of metformin are not explicitly stated in the provided documents. However, there are warnings and precautions related to certain conditions and situations that may require ca..."
3,What are the common adverse reactions of amlodipine?,"Amlodipine_label.pdf, Atorvastatin_label.pdf",yes,The most commonly reported side effects more frequent than placebo are dizziness and edema. [2]
4,What are the contraindications of atorvastatin?,Atorvastatin_label.pdf,yes,"The contraindications of atorvastatin are not explicitly stated in the provided documents. However, there are warnings and precautions listed in the labeling, such as liver enzyme testing, monitor..."
5,What are the warnings and precautions associated with ciprofloxacin?,"Azithromycin_label.pdf, Ciprofloxacin_label.pdf",yes,"According to the provided context, the warnings and precautions associated with ciprofloxacin are:\n\n5.1 Hypersensitivity Reactions\n5.2 Potential for Microbial Overgrowth with Prolonged Use\n5.3..."
6,What are the contraindications of lisinopril?,Lisinopril_label.pdf,yes,I could not find this in the provided documents.
7,What are the contraindications of losartan?,Losartan_label.pdf,yes,"Losartan potassium is contraindicated in patients who are hypersensitive to any component of this product [3]. Additionally, losartan potassium is contraindicated for coadministration with aliskir..."
8,What are the indications and usage of omeprazole?,"Amoxicillin_label.pdf, Lisinopril_label.pdf, Omeprazole_label.pdf",yes,The indications and usage of omeprazole are as follows:\n\nTreats frequent heartburn (occurs 2 or more days a week) not intended for immediate relief of heartburn; this drug may take 1 to 4 days f...
9,What are the common adverse reactions of sertraline?,"Amlodipine_label.pdf, Azithromycin_label.pdf, Lisinopril_label.pdf, Losartan_label.pdf",no,I could not find this in the provided documents.


In [ ]:
eval_df["grounded"] = ["?"] * len(eval_df)
eval_df["correct"] = ["?"] * len(eval_df)

eval_df.to_csv(PROJECT_ROOT / "evaluation_results.csv", index=False)

# Compact version to paste into your README
readme_table = eval_df.assign(answer=eval_df["answer"].str.slice(0, 120) + "...")
print(readme_table[["question", "retrieved_source", "answer", "grounded", "correct"]].to_markdown(index=False))

| question                                                                    | retrieved_source                                                                          | answer                                                                                                                      | grounded   | correct   |
|:----------------------------------------------------------------------------|:------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------|:-----------|:----------|
| What are the Drug/Laboratory Test Interactions of Amoxicillin?              | Amoxicillin_label.pdf                                                                     | Unfortunately, I could not find any information on Drug/Laboratory Test Interactions of Amoxicillin in the provided docu... | ?          | ?         |
| What are the clinical conside

## 7. Export
The vector store is already saved in `backend/data/vector_store/`. Here we also save a `config.json` so the backend uses **exactly** the same embedding model and collection name (no rebuilding at request time).

In [76]:
config = {
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "min_score": MIN_SCORE,
    "num_chunks": len(chunks),
    "num_documents": len({c["source"] for c in chunks}),
}
(EXPORT_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

size_mb = sum(f.stat().st_size for f in EXPORT_DIR.rglob("*") if f.is_file()) / 1e6
print(json.dumps(config, indent=2))
print(f"\nVector store folder: {EXPORT_DIR}")
print(f"Total size: {size_mb:.1f} MB  (commit it to GitHub only if it is well under ~50 MB)")

{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "collection_name": "documents",
  "chunk_size": 1200,
  "chunk_overlap": 250,
  "top_k": 6,
  "min_score": 0.35,
  "num_chunks": 730,
  "num_documents": 10
}

Vector store folder: d:\rag-assistant-starter-kit\rag-assistant-project\backend\data\vector_store
Total size: 17.0 MB  (commit it to GitHub only if it is well under ~50 MB)
